# CA26 — Experiment Notebook (multi-seed aggregation)

This notebook shows a *lightweight* experimental workflow for CA26:
- Run short experiments for multiple seeds (default: 3 seeds)
- Save a small `metadata.json` per run (seed, config, final metric, timestamp)
- Aggregate results into `outputs/aggregate.csv` and produce an aggregated plot (mean ± std)

Notes:
- This notebook is **non-executed** by default and is designed to be a reproducible recipe for small runs (use `epochs <= 5` for quick runs or CI).
- Running this notebook will call `scripts.run_experiment.fit(cfg, out_dir)` which is import-safe; keep runs short for CI.

## 1. Imports and helpers

Import necessary packages and define small helper utilities used by the notebook.

In [ ]:
from __future__ import annotations

from dataclasses import asdict
from datetime import datetime
from pathlib import Path
import json
import shutil
import sys
from typing import Sequence

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Local project imports
from src.config import load_config, ExperimentConfig
from scripts.run_experiment import fit


def write_metadata(out_dir: Path, seed: int, cfg: ExperimentConfig, final_loss: float | None):
    meta = {
        "seed": int(seed),
        "config": asdict(cfg),
        "final_loss": float(final_loss) if final_loss is not None else None,
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "commit": None  # Optionally fill with git commit SHA if available
    }
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "metadata.json").write_text(json.dumps(meta, indent=2))
    return meta

## 2. Single-run helper

`run_single` runs `fit()` with provided `ExperimentConfig` and writes `metadata.json`. It keeps each run in `outputs/seed-<s>`.

In [ ]:
def run_single(cfg: ExperimentConfig, seed: int, out_root: Path, make_clean: bool = True) -> dict:
    """Run a single short experiment and write metadata.json.

    Parameters
    - cfg: ExperimentConfig (modified in-place with seed and short epochs)
    - seed: random seed
    - out_root: base outputs directory (per-run dir = out_root/seed-<seed>)
    - make_clean: if True, remove existing out dir before running
    """
    out_dir = Path(out_root) / f"seed-{seed}"
    if make_clean and out_dir.exists():
        shutil.rmtree(out_dir)

    # Ensure deterministic seeds set by train config; fit() will call set_seed
    cfg.train.seed = int(seed)

    # Run a short fit and capture results
    res = fit(cfg, out_dir)
    final_loss = res.losses[-1] if res.losses else None

    # Write metadata.json
    meta = write_metadata(out_dir, seed, cfg, final_loss)
    return {"seed": seed, "final_loss": final_loss, "out_dir": str(out_dir)}

## 3. Aggregation helper

`aggregate_runs` looks for `metadata.json` files in `out_root/seed-*`, builds a DataFrame, writes `aggregate.csv` and creates an aggregated plot (mean ± std of final loss).

In [ ]:
def aggregate_runs(out_root: Path, out_aggregate_csv: Path | None = None, out_plot: Path | None = None) -> pd.DataFrame:
    out_root = Path(out_root)
    rows = []
    for p in sorted(out_root.glob("seed-*")):
        m = p / "metadata.json"
        if not m.exists():
            continue
        meta = json.loads(m.read_text())
        rows.append({"seed": meta.get("seed"), "final_loss": meta.get("final_loss"), "timestamp": meta.get("timestamp")})

    if not rows:
        raise RuntimeError("No run metadata found under " + str(out_root))

    df = pd.DataFrame(rows).sort_values("seed")
    if out_aggregate_csv is None:
        out_aggregate_csv = out_root / "aggregate.csv"
    out_aggregate_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_aggregate_csv, index=False)

    # Plot aggregated final loss (bar + error)
    means = df["final_loss"].astype(float).mean()
    stds = df["final_loss"].astype(float).std()
    if out_plot is None:
        out_plot = out_root / "aggregate_loss.png"
    fig, ax = plt.subplots()
    ax.bar([0], [means], yerr=[stds], capsize=6)
    ax.set_xticks([0])
    ax.set_xticklabels(["final_loss (mean±std)"])
    ax.set_ylabel("loss")
    fig.tight_layout()
    fig.savefig(out_plot, dpi=150)
    plt.close(fig)

    return df

## 4. Example: Run 3 seeds (non-executed cell)

The following cell demonstrates how to run seeds [0,1,2] with a short config (epochs=5) and then aggregate results. Do not execute this cell automatically in CI unless you intend to run the short experiment.

In [ ]:
# Example (non-executed by default):
#
# from src.config import load_config
# cfg = load_config("configs/default.yaml")
# cfg.train.epochs = 5  # keep short for demos
# cfg.train.batch_size = 64
# out_root = Path("outputs/ca26_experiment")
# seeds = [0, 1, 2]
#
# # Run the seeds
# results = []
# for s in seeds:
#     r = run_single(cfg, s, out_root)
#     results.append(r)
#
# # Aggregate and inspect
# df = aggregate_runs(out_root)
# print(df)
#
# # Sample: show aggregate.csv location
# print("Wrote:", out_root / "aggregate.csv")

## 5. Plot example: load per-run loss curves

If you want to show individual run loss curves, `scripts.run_experiment.fit` writes `loss/loss_curve.png` under each run's output directory. The cell below demonstrates how to load and display them programmatically (non-executed):

In [ ]:
# Visualization helper (non-executed example):
#
# import matplotlib.image as mpimg
# fig, axs = plt.subplots(1, len(seeds), figsize=(4*len(seeds), 3))
# for i, s in enumerate(seeds):
#     img_path = out_root / f"seed-{s}" / "loss" / "loss_curve.png"
#     if not img_path.exists():
#         continue
#     img = mpimg.imread(img_path)
#     ax = axs[i] if len(seeds) > 1 else axs
#     ax.imshow(img)
#     ax.axis('off')
# fig.tight_layout()
# fig.savefig(out_root / "all_loss_curves.png", dpi=150)
# plt.close(fig)

## 6. Notes & reproducibility

- For CI or quick smoke tests, set `cfg.train.epochs = 1` and run a single seed.
- To be fully reproducible, record the commit SHA and the pip freeze of the environment in `metadata.json` (this notebook writes a `commit` field placeholder).
- Use `scripts/aggregate_results.py` (or the `aggregate_runs` helper above) to build `outputs/aggregate.csv` used in the report.

---

*End of notebook.*

This notebook is a compact, reproducible recipe for running small multi-seed experiments for CA26. Adjust `epochs`, `batch_size`, and `seeds` to match your compute budget and desired confidence in results.